In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import joblib

In [2]:
df = pd.read_csv("../data/creditwise_raw_data.csv")
print("Dataset shape:", df.shape)

Dataset shape: (1000, 20)


In [3]:
labeled_df = df[df["Loan_Approved"].notna()].copy()
unlabeled_df = df[df["Loan_Approved"].isna()].copy()

print("Labeled rows:", len(labeled_df))
print("Unlabeled rows:", len(unlabeled_df))

Labeled rows: 950
Unlabeled rows: 50


In [4]:
X_full = labeled_df.drop(
    columns=["Loan_Approved", "Applicant_ID"]
)

y_full = labeled_df["Loan_Approved"].map({
    "No": 0,
    "Yes": 1
})

In [5]:
unlabeled_ids = unlabeled_df["Applicant_ID"].copy()

X_unlabeled = unlabeled_df.drop(
    columns=["Loan_Approved", "Applicant_ID"]
)

In [6]:
def add_features(df):
    df = df.copy()

    df["DTI_Ratio_sq"] = df["DTI_Ratio"] ** 2
    df["Credit_Score_sq"] = df["Credit_Score"] ** 2
    df["Applicant_Income_log"] = np.log1p(df["Applicant_Income"])

    return df

X_full = add_features(X_full)
X_unlabeled = add_features(X_unlabeled)

In [7]:
features_to_remove = [
    "Savings",
    "Collateral_Value",
    "Existing_Loans",
    "Marital_Status",
    "Loan_Term",
    "Age",
    "Dependents"
]

X_full = X_full.drop(columns=features_to_remove)
X_unlabeled = X_unlabeled.drop(columns=features_to_remove)

print("Final training shape:", X_full.shape)
print("Final prediction shape:", X_unlabeled.shape)

Final training shape: (950, 14)
Final prediction shape: (50, 14)


In [8]:
cat_cols_final = X_full.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols_final = X_full.select_dtypes(
    include=["number"]
).columns.tolist()

In [9]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        drop="first",
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor_final = ColumnTransformer([
    ("num", num_pipeline, num_cols_final),
    ("cat", cat_pipeline, cat_cols_final)
])

In [10]:
final_model = Pipeline([
    ("preprocessor", preprocessor_final),
    ("model", LogisticRegression(
        C=100,
        max_iter=1000
    ))
])

In [11]:
final_model.fit(X_full, y_full)
print("Final model trained successfully.")

Final model trained successfully.


In [13]:
joblib.dump(
    final_model,
    "../models/creditwise_logistic_regression.pkl"
)

print("Final model saved successfully.")

Final model saved successfully.


In [14]:
y_unlabeled_pred = final_model.predict(X_unlabeled)

y_unlabeled_prob = final_model.predict_proba(
    X_unlabeled
)[:, 1]

print("Predictions generated successfully.")

Predictions generated successfully.


## Final Predictions

In [15]:
prediction_df = pd.DataFrame({
    "Applicant_ID": unlabeled_ids,
    "Loan_Approved_Prediction": np.where(
        y_unlabeled_pred == 1,
        "Yes",
        "No"
    ),
    "Approval_Probability": y_unlabeled_prob
})

prediction_df["Rejection_Probability"] = (
    1 - prediction_df["Approval_Probability"]
)

prediction_df.head(10)

,Applicant_ID,Loan_Approved_Prediction,Approval_Probability,Rejection_Probability
15,16.0,No,2.956034e-02,0.970440
26,27.0,No,4.040938e-03,0.995959
100,101.0,No,1.392147e-02,0.986079
122,123.0,Yes,6.945149e-01,0.305485
191,NaN,No,8.716737e-03,0.991283
195,196.0,No,1.720254e-01,0.827975
263,264.0,No,2.193543e-09,1.000000
290,291.0,Yes,9.701062e-01,0.029894
293,294.0,No,1.881819e-03,0.998118
310,311.0,No,6.913761e-02,0.930862


In [16]:
prediction_df.to_csv(
    "../data/final_loan_predictions.csv",
    index=False
)

print("Predictions saved successfully.")

Predictions saved successfully.


In [18]:
print("Number of predictions:", len(prediction_df))

print("\nPrediction counts:")
print(prediction_df["Loan_Approved_Prediction"].value_counts())

print("\nPrediction table:")
# display(prediction_df)

Number of predictions: 50

Prediction counts:
Loan_Approved_Prediction
No     31
Yes    19
Name: count, dtype: int64

Prediction table:
